# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [43]:
from litellm import completion
from dotenv import load_dotenv
from pricer.items import Item
from tqdm import tqdm
import os
import json

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [44]:
LITE_MODE = True

MODEL = "ollama/llama3.2:1b"
API_BASE = "http://localhost:11434"

username = "andrevsilva"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

In [45]:
train, val, test = Item.from_hub(dataset)

N = 10
ntrain = int(0.9 * N)
nval = int(0.05 * N )
ntest = int(0.05 * N)

print(f"Train: {ntrain}")
print(f"Val: {nval}")
print(f"Test: {ntest}")

items = train[ :ntrain] + val[ :nval] + test[ :ntest]

print(f"Loaded {len(items):,} items")
print(items[0])

Train: 9
Val: 0
Test: 0
Loaded 9 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbol

In [46]:
print(items[2].id)

None


In [47]:
for index, item in enumerate(items):
    item.id = index

In [48]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [49]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [50]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]

response = completion(
    messages=messages,
    model="ollama/llama3.2:1b"
)

print(response.choices[0].message.content)
print()

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")

**Rewritten Schlage F59 AND 613 Andover Interior Knob with Deadbolt**
The Schlage F59 AND 613 Andover Interior Knob provides a secure and stylish deadbolt option for front door security.

**Details**: This non-handed knob style has a 4" minimum center to center door prep required for installation. The Schlage handleset offers a lifetime mechanical and finish warranty, ensuring long-term reliability.

Input tokens: 406
Output tokens: 87


In [51]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2:1b", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Here is a concise description of the product in the requested format:

Schlage F59 AND 613 Andover Interior Knob with Deadbolt
The Schlage F59 AND 613 andAndover Interior Knob with Deadbolt provides security and style for the front door, featuring an oil rubbed bronze finish.

This two-piece handle set includes a deadbolt lock and non-handed knob design.

Input tokens: 406
Output tokens: 79
Cost: 0.000 cents


In [52]:
MODEL = "ollama/llama3.2:1b"

In [53]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [54]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [55]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "ollama/llama3.2:1b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece m

In [56]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [57]:
make_file(0, N-1, "jsonl/0_1000.jsonl")

In [58]:
def generate_summary(item):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item.full}
    ]

    response = completion(
        model=MODEL,
        messages=messages,
        api_base=API_BASE,
        temperature=0.2
    )

    return response.choices[0].message.content

In [59]:
missing = [item.id for item in items if not getattr(item, "summary", None)]
print("Missing summaries:", missing[:20])
print("Total missing:", len(missing))

Missing summaries: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Total missing: 9


In [60]:
print(items[10].full)

IndexError: list index out of range

In [ ]:
for item in items:
    item.full = None
    item.id = None

In [ ]:
if LITE_MODE:
    train = items[:ntrain]
    val = items[ntrain:ntrain+nval]
    test = items[ntrain+nval:ntrain+nval+ntest]

In [ ]:
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    Item.push_to_hub(lite, train, val, test)
else:
    Item.push_to_hub(full, train, val, test)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/712 [00:00<?, ?B/s]